In [1]:
!pip install -q crewai



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46

In [2]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from crewai import LLM
llm = LLM(
    model="gemini/gemini-flash-lite-latest",
    temperature=0.2
    )

In [3]:
issues = [
    {"title": "App crashes when I click Save",
     "body": "Every time I click Save on the profile page, the app closes. "
             "Started after yesterday's update. Android 14."},
    {"title": "Please add export to PDF",
     "body": "It would be great to download my monthly report as a PDF."},
    {"title": "How do I reset my password?",
     "body": "I forgot my password and can't find the option."},
]

docs = """App help docs:
- Reset password: go to Login > 'Forgot password' and follow the email link.
- Change email: Settings > Account > Email.
- Reports can be downloaded as CSV from Reports > Download."""

In [4]:
from crewai import Agent, Task, Crew

async def run_crew(role, goal, instructions, issue_text):
  agent = Agent(
      role = role,
      goal = goal,
      backstory = "You are an experienced " + role +". You are clear and practical. ",
      llm = llm,
      verbose = False,
  )
  task = Task(
      description = instructions + "\n\nIssue:\n" + issue_text,
      expected_output = "A short, clear response in plain text.",
      agent = agent,
  )
  crew = Crew(
      agents = [agent],
      tasks = [task],
      verbose = False
  )
  result = await crew.kickoff_async()
  return result.raw

In [5]:
answer = await run_crew(
    role="QA Engineer",
    goal="Assess bugs clearly",
    instructions="Give the severity (critical, high or low) and 3 steps to reproduce this bug.",
    issue_text=issues[0]["title"] + "\n" + issues[0]["body"],
)
print(answer)

Severity: Critical

Steps to reproduce:
1. Open the app and navigate to the profile page.
2. Make any change to the profile information.
3. Click the "Save" button.


In [6]:
from crewai.flow.flow import Flow, start, listen, router, or_

class TriageFlow(Flow):

    @start()
    def read_issue(self):
        self.state["text"] = self.state["title"] + "\n" + self.state["body"]
        print("\nIssue:", self.state["title"])

    @listen(read_issue)
    def classify_issue(self):
        answer = llm.call(
            "Classify this issue into one category:\n"
            "- bug: something is broken, crashing or not working as expected\n"
            "- feature: a request for something new or an improvement\n"
            "- question: the user is asking how to do something\n"
            "Reply with one word only.\n\nIssue:\n" + self.state["text"]
        )
        self.state["type"] = answer.strip().lower()
        print("Type:", self.state["type"])

    @router(classify_issue)
    def route_type(self):
        if self.state["type"] in ["bug", "feature", "question"]:
            return self.state["type"]
        return "question"

    @listen("bug")
    async def handle_bug(self):
        self.state["label"] = "bug"
        self.state["comment"] = await run_crew(
            role="QA Engineer",
            goal="Assess bugs clearly",
            instructions="Give the severity (critical, high or low) and 3 steps to reproduce this bug.",
            issue_text=self.state["text"],
        )

    @listen("feature")
    async def handle_feature(self):
        self.state["label"] = "feature"
        self.state["comment"] = await run_crew(
            role="Product Manager",
            goal="Turn feature requests into clear user stories",
            instructions="Write a user story in the form 'As a <user>, I want <goal>, "
                         "so that <benefit>', followed by 2 acceptance criteria.",
            issue_text=self.state["text"],
        )

    @listen("question")
    async def handle_question(self):
        self.state["label"] = "question"
        self.state["comment"] = await run_crew(
            role="Support Agent",
            goal="Answer user questions accurately",
            instructions="Answer the user's question using ONLY these docs. If the "
                         "answer is not in the docs, say a team member will follow up.\n\n"
                         + docs,
            issue_text=self.state["text"],
        )

    @listen(or_(handle_bug, handle_feature, handle_question))
    def post_reply(self):
        print("Label:", self.state["label"])
        print("Comment:\n" + self.state["comment"])
        return self.state["comment"]

In [7]:
for issue in issues:
    flow = TriageFlow()
    await flow.kickoff_async(inputs=issue)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: TriageFlow                                                                                               │
│  ID: c5626cda-a3d0-4189-a84d-b3396565ac86                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: TriageFlow                                                                                               │
│  ID: c5626cda-a3d0-4189-a84d-b3396565ac86                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: read_issue                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Issue: App crashes when I click Save


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: read_issue                                                                                             │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: handle_question                                                                                        │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Label: question
Comment:
Go to Login > 'Forgot password' and follow the email link.


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: post_reply                                                                                             │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: post_reply                                                                                             │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: TriageFlow                                                                                               │
│  ID: eccea98d-be5a-4fba-8737-0e50180f42b9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯